In [0]:
#Installs the Google Ads Python client library using pip. This is required to interact with the Google Ads API.
%pip install google-ads

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
Restarts the Python kernel to ensure that newly installed packages are available in the environment.
"""
%restart_python

In [0]:
#Creates the 'googleads_bronze' database if it does not already exist.
spark.sql("CREATE DATABASE IF NOT EXISTS googleads_bronze")
spark.catalog.listDatabases()

[Database(name='default', catalog='workspace', description='Default schema (auto-created)', locationUri=''),
 Database(name='googleads_bronze', catalog='workspace', description='', locationUri=''),
 Database(name='gotomeeting_bronze', catalog='workspace', description='', locationUri=''),
 Database(name='gotomeeting_gold', catalog='workspace', description='', locationUri=''),
 Database(name='gotomeeting_silver', catalog='workspace', description='', locationUri=''),
 Database(name='information_schema', catalog='workspace', description='Information schema (auto-created)', locationUri='')]

In [0]:
# Import necessary modules from the Google Ads library for API interaction
# and error handling.
from google.ads.googleads.client import GoogleAdsClient
from google.ads.googleads.errors import GoogleAdsException

# Import datetime libraries for dynamic date range calculations.
from datetime import datetime, timedelta, date

# Import typing for type hinting, improving code clarity and maintainability.
from typing import Optional, Dict

# Import the logging module to enable detailed output for monitoring and debugging.
import logging
# Optional logging
#logging.basicConfig(level=logging.INFO)

# Set up basic logging to capture informational messages during execution.
# This helps in tracking the script's progress and diagnosing issues.
# The log level is set to INFO, which provides a good balance of detail.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Time window: Last 30 days
#start_date="2024-07-30"
#end_date = datetime.today().strftime("%Y-%m-%d")
#start_date = (datetime.today() - timedelta(days=30)).strftime("%Y-%m-%d")
# Log the selected date range for verification purposes.
#logging.info(f"Reporting window set from {start_date} to {end_date}.")

In [0]:
# Remove all widgets from the notebook UI to clean up after capturing values.
dbutils.widgets.removeAll()
# Create a text widget for the start date, defaulting to 7 days ago. This allows users to select the reporting window's beginning.
dbutils.widgets.text(
"start_date",
    "2024-07-01",
    "Start date (YYYY-MM-DD)"
)

# Create a text widget for the end date, defaulting to today. This sets the reporting window's end.
dbutils.widgets.text(
    "end_date",
    date.today().isoformat(),
    "End date (YYYY-MM-DD)"
)

# Create a text widget for the Google Ads customer ID, defaulting to a sample value. This identifies the account to query.
dbutils.widgets.text(
    "customer_id",
    "1401815809",
    "Customer ID (e.g. 1234567890)"
)

# Retrieve the start date value entered by the user from the widget.
start_date = dbutils.widgets.get("start_date")

# Retrieve the end date value entered by the user from the widget.
end_date = dbutils.widgets.get("end_date")

# Retrieve the customer ID value entered by the user from the widget.
customer_id = dbutils.widgets.get("customer_id")

def parse_iso_date(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d").date()
    except Exception:
        raise ValueError(f"Invalid date format: {s}. Required YYYY-MM-DD")

start = parse_iso_date(start_date)
end = parse_iso_date(end_date)

if start > end:
    raise ValueError("start_date must be <= end_date")

logging.info(f"Validated date range: {start} → {end}")

2026-03-03 06:46:03,972 - INFO - Received command c on object id p0
2026-03-03 06:46:04,013 - INFO - Validated date range: 2024-07-01 → 2026-02-23


In [0]:
# Load credentials from secrets + optional override widget

# Add widget for optional refresh token override
# dbutils.widgets.text("refresh_token_override", "", "Refresh Token Override (Optional)")
# refresh_token_override = dbutils.widgets.get("refresh_token_override")

try:
    #developer_token   = dbutils.secrets.get(scope="googleads", key="developer_token")
    #client_id         = dbutils.secrets.get(scope="googleads", key="client_id")
    #client_secret     = dbutils.secrets.get(scope="googleads", key="client_secret")
    #refresh_token     = dbutils.secrets.get(scope="googleads", key="refresh_token")
    #login_customer_id = dbutils.secrets.get(scope="googleads", key="login_customer_id")
    developer_token = "iNtyq5-pv-k9S6UVt83QTA"
    client_id = "728675242127-dak81mosr3utt0k41f6njm8vgit1ub6l.apps.googleusercontent.com"
    client_secret = "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"
    refresh_token = "1//0gYeHBnDvmUsyCgYIARAAGBASNwF-L9Ir-wJWFTu5GnVP0U6-qVseGVoJJzBdhv4aQJZxYf3KEUatKBEGUz0TcOm5Kl_rlDzdoqk"
    login_customer_id = "1854910982"
except Exception as e:
    raise RuntimeError("❌ Missing secrets in scope 'googleads'. Please add: developer_token, client_id, client_secret, refresh_token, login_customer_id") from e

# If widget provided, override the secret refresh token just for this run
#if refresh_token_override.strip():
#    print("⚠️ Using refresh token override provided in widget (not saved to secrets).")
#    refresh_token = refresh_token_override.strip()

config = {
    "developer_token": developer_token,
    "client_id": client_id,
    "client_secret": client_secret,
    "refresh_token": refresh_token,
    "login_customer_id": login_customer_id,
    "use_proto_plus": True
}

try:
    client = GoogleAdsClient.load_from_dict(config)
except Exception as e:
    msg = str(e).lower()
    if "refresh token" in msg or "invalid_grant" in msg or "expired" in msg:
        raise RuntimeError(
            "⚠️ Refresh token expired. Either:\n"
            "1. Paste a new token into the 'refresh_token_override' widget for this run, OR\n"
            "2. Permanently update the 'refresh_token' in Databricks secrets (scope: googleads)."
        )
    else:
        raise


In [0]:
# Option 1: Set inline manually
#developer_token = "iNtyq5-pv-k9S6UVt83QTA"
#client_id = "728675242127-dak81mosr3utt0k41f6njm8vgit1ub6l.apps.googleusercontent.com"
#client_secret = "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"
#refresh_token = "1//0gFqRJRYJC3r9CgYIARAAGBASNwF-L9Ir2EthZqgalMLfjnP-o-KoU73Ncp34bGV7mmYTlwXud-xmn3CnG-b1wNdx4acFyR2GBM4"
#login_customer_id = "1854910982"
#customer_id = "1401815809"         # Required

# Option 2: (Commented for now) Fetch from Databricks secrets
# developer_token = dbutils.secrets.get(scope="googleads", key="developer_token")
# client_id       = dbutils.secrets.get(scope="googleads", key="client_id")
# client_secret   = dbutils.secrets.get(scope="googleads", key="client_secret")
# refresh_token   = dbutils.secrets.get(scope="googleads", key="refresh_token")

# Load Google Ads config into client manually
'''config = {
    "developer_token": developer_token,
    "client_id": client_id,
    "client_secret": client_secret,
    "refresh_token": refresh_token,
    "login_customer_id": login_customer_id,
    "use_proto_plus": True
}

client = GoogleAdsClient.load_from_dict(config)
print(f"Date Window: {start_date} → {end_date}, Customer: {customer_id}")
'''

'config = {\n    "developer_token": developer_token,\n    "client_id": client_id,\n    "client_secret": client_secret,\n    "refresh_token": refresh_token,\n    "login_customer_id": login_customer_id,\n    "use_proto_plus": True\n}\n\nclient = GoogleAdsClient.load_from_dict(config)\nprint(f"Date Window: {start_date} → {end_date}, Customer: {customer_id}")\n'

In [0]:
# Remove all widgets from the notebook UI to clean up after capturing values.
dbutils.widgets.removeAll()

In [0]:
"""
Defines a dictionary of GAQL queries for different Google Ads reporting needs. Each key represents a report type and the value is the corresponding query string.
"""
query_map = {
    "core_campaign_performance": f"""
        SELECT
            campaign.id,
            segments.date,
            campaign.name,
            campaign.status,
            campaign.advertising_channel_type,
            metrics.impressions,
            metrics.clicks,
            metrics.ctr,
            metrics.cost_micros,
            metrics.average_cpc
        FROM campaign
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY metrics.impressions DESC
    """,
    "campaign_asset": f"""
        SELECT
            campaign.id,
            campaign.name,
            campaign.status,
            asset.id,
            asset.type,
            campaign_asset.status,
            segments.date,
            segments.device,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions,
            metrics.ctr,
            metrics.average_cpc
        FROM campaign_asset
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.impressions DESC
    """,
    "ad_group_ad_asset_view": f"""
        SELECT
            campaign.id,
            campaign.name,
            ad_group.id,
            ad_group.name,
            ad_group_ad_asset_view.asset,
            ad_group_ad_asset_view.field_type,
            segments.date,
            segments.device,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions,
            metrics.ctr,
            metrics.average_cpc
        FROM ad_group_ad_asset_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.impressions DESC
    """,
    "asset": f"""
        SELECT
            asset.id,
            asset.type,
            asset.source,
            asset.text_asset.text,
            asset.image_asset.full_size.url,
            asset.sitelink_asset.link_text,
            asset.callout_asset.callout_text,
            asset.structured_snippet_asset.header,
            asset.structured_snippet_asset.values
        FROM asset
        ORDER BY asset.type
    """,
    "search_keyword_performance": f"""
        SELECT
            campaign.id,
            ad_group.id,
            ad_group_criterion.criterion_id,
            segments.date,
            campaign.name,
            ad_group.name,
            ad_group_criterion.keyword.text,
            ad_group_criterion.keyword.match_type,
            ad_group_criterion.quality_info.quality_score,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros
        FROM keyword_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND ad_group_criterion.status = 'ENABLED'
        ORDER BY metrics.clicks DESC
    """,
    "search_term_analysis": f"""
        SELECT
            campaign.id,
            ad_group.id,
            segments.date,
            campaign.name,
            ad_group.name,
            search_term_view.search_term,
            search_term_view.status,
            segments.search_term_match_type,
            segments.search_term_match_source,
            segments.keyword.ad_group_criterion,
            segments.keyword.info.text,
            segments.keyword.info.match_type,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions,
            metrics.average_cpc,
            metrics.ctr
        FROM search_term_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.clicks > 0
        ORDER BY metrics.clicks DESC
    """,
    "conversion_action":f"""
        SELECT
            conversion_action.id,
            conversion_action.name,
            conversion_action.status,
            conversion_action.category,
            conversion_action.primary_for_goal,
            conversion_action.counting_type,
            conversion_action.attribution_model_settings.attribution_model,
            conversion_action.click_through_lookback_window_days,
            conversion_action.view_through_lookback_window_days,
            conversion_action.include_in_conversions_metric
        FROM conversion_action
    """,
    "conversion_performance": f"""
        SELECT
            campaign.id,
            segments.date,
            campaign.name,
            campaign.status,
            segments.device,
            segments.conversion_action,
            segments.conversion_action_name,
            segments.conversion_action_category,
            metrics.conversions,
            metrics.conversions_value,
            metrics.all_conversions,
            metrics.all_conversions_value,
            metrics.value_per_conversion,
            metrics.value_per_all_conversions
        FROM campaign
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.all_conversions > 0
        ORDER BY metrics.all_conversions_value DESC
    """,
    "ad_copy_and_landing_page_performance": f"""
        SELECT
            campaign.id,
            ad_group.id,
            ad_group_ad.ad.id,
            segments.date,
            campaign.name,
            ad_group.name,
            ad_group_ad.ad.final_urls,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions
        FROM ad_group_ad
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.clicks DESC
    """,
    "optimization_device_and_time": f"""
        SELECT
            campaign.id,
            segments.date,
            segments.device,
            segments.day_of_week,
            segments.hour,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions,
            metrics.cost_micros,
            metrics.ctr,
            metrics.average_cpc
        FROM campaign
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.clicks DESC
    """,
    "placement_performance": f"""
        SELECT
            campaign.id,
            campaign.name,
            campaign.advertising_channel_type,
            ad_group.id,
            ad_group.name,
            segments.date,
            detail_placement_view.placement,
            detail_placement_view.placement_type,
            detail_placement_view.display_name,
            detail_placement_view.group_placement_target_url,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions,
            metrics.ctr,
            metrics.average_cpc
        FROM detail_placement_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.cost_micros DESC
    """,
    "group_placement_performance": f"""
        SELECT
            campaign.id,
            campaign.name,
            campaign.advertising_channel_type,
            ad_group.id,
            ad_group.name,
            segments.date,
            group_placement_view.placement,
            group_placement_view.placement_type,
            group_placement_view.display_name,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions,
            metrics.ctr,
            metrics.average_cpc   
        FROM group_placement_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.cost_micros DESC
    """,
    "audience_gender_performance": f"""
        SELECT
            campaign.id,
            ad_group.id,
            ad_group_criterion.criterion_id,
            segments.date,
            campaign.name,
            ad_group_criterion.gender.type,
            metrics.impressions,
            metrics.clicks,
            metrics.cost_micros,
            metrics.conversions
        FROM gender_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.clicks DESC
    """,
    "audience_age_performance": f"""
        SELECT
            campaign.id,
            ad_group.id,
            ad_group_criterion.criterion_id,
            segments.date,
            campaign.name,
            ad_group_criterion.age_range.type,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions,
            metrics.cost_micros
        FROM age_range_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND metrics.impressions > 0
        ORDER BY metrics.clicks DESC
    """,
    #"audience_interest_performance": f"""
    #    SELECT
    #        campaign.id,
    #        campaign.name,
    #        ad_group.id,
    #        ad_group_criterion.criterion_id,
    #        ad_group_criterion.display_name,
    #        ad_group_criterion.type,
    #        metrics.impressions,
    #        metrics.clicks,
    #        metrics.conversions
    #    FROM ad_group_audience_view
    #    WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
    #        AND metrics.impressions > 0
    #    ORDER BY metrics.clicks DESC
    #""",
    "competitive_impression_share": f"""
        SELECT
            campaign.id,
            segments.date,
            campaign.name,
            metrics.search_impression_share,
            metrics.search_top_impression_share,
            metrics.search_absolute_top_impression_share
        FROM campaign
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
          AND campaign.advertising_channel_type = 'SEARCH'
        ORDER BY metrics.search_impression_share DESC
    """,
    "network_performance": f"""
        SELECT
            campaign.id,
            segments.date,
            campaign.name,
            segments.ad_network_type,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions
        FROM campaign
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY metrics.clicks DESC
    """,
    "display_ad_viewability": f"""
        SELECT
            segments.date,
            campaign.name,
            ad_group.name,
            metrics.active_view_impressions,
            metrics.active_view_measurability,
            metrics.active_view_viewability,
            metrics.active_view_cpm
        FROM ad_group
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
        AND campaign.advertising_channel_type IN ('DISPLAY', 'PERFORMANCE_MAX')
        AND metrics.active_view_impressions > 0
    """,
    "Call_lead_generation": f"""
        SELECT
            segments.date,
            campaign.name,
            ad_group.name,
            metrics.phone_calls,
            metrics.phone_impressions,
            metrics.phone_through_rate
        FROM ad_group
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
        AND metrics.phone_impressions > 0
    """,
    "geographic_performance": f"""
        SELECT
            campaign.id,
            segments.date,
            geographic_view.country_criterion_id,
            segments.geo_target_city,
            geographic_view.location_type,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions,
            metrics.cost_micros,
            metrics.ctr
        FROM geographic_view
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}' 
        AND metrics.impressions > 0
        ORDER BY metrics.clicks DESC
        """,
    "ads_performance":f"""
        SELECT
            ad_group.id,
            ad_group_ad.ad.id,
            metrics.impressions,
            metrics.clicks,
            metrics.conversions,
            metrics.cost_micros
        FROM ad_group_ad
        WHERE segments.date BETWEEN '{start_date}' AND '{end_date}'
        """,
    "ads_data":f"""
        SELECT
            campaign.id,
            campaign.name,
            ad_group.id,
            ad_group.name,
            ad_group_ad.ad.id,
            ad_group_ad.ad.type,
            ad_group_ad.status,
            ad_group_ad.ad.final_urls,

            -- Responsive Search Ad fields
            ad_group_ad.ad.responsive_search_ad.headlines,
            ad_group_ad.ad.responsive_search_ad.descriptions,

            -- Expanded Text Ad fields
            ad_group_ad.ad.expanded_text_ad.headline_part1,
            ad_group_ad.ad.expanded_text_ad.headline_part2,
            ad_group_ad.ad.expanded_text_ad.headline_part3,
            ad_group_ad.ad.expanded_text_ad.description,
            ad_group_ad.ad.expanded_text_ad.description2,

            -- Call Ad fields
            --ad_group_ad.ad.call_ad.business_name,
            --ad_group_ad.ad.call_ad.country_code,
            --ad_group_ad.ad.call_ad.phone_number,
            --ad_group_ad.ad.call_ad.headline1,
            --ad_group_ad.ad.call_ad.headline2,
            --ad_group_ad.ad.call_ad.description1,
            --ad_group_ad.ad.call_ad.description2,

            -- Responsive Display Ad fields
            ad_group_ad.ad.responsive_display_ad.long_headline,
            ad_group_ad.ad.responsive_display_ad.headlines,
            ad_group_ad.ad.responsive_display_ad.descriptions,

            -- Image Ad fields
            ad_group_ad.ad.image_ad.image_url,
            ad_group_ad.ad.image_ad.mime_type
        FROM ad_group_ad
        """,
    "geo_target_constant": f"""
        SELECT
            geo_target_constant.id,
            geo_target_constant.name,
            geo_target_constant.canonical_name,
            geo_target_constant.country_code,
            geo_target_constant.target_type
        FROM geo_target_constant
        """
}


2026-03-03 06:46:05,202 - INFO - Received command c on object id p0


In [0]:
"""
Validates a GAQL query using the Google Ads API in validate_only mode. It checks the syntax and structure of the query without executing it.
"""
def validate_gaql(client: GoogleAdsClient, query: str) -> bool:
    """
    Validate a GAQL (Google Ads Query Language) statement without executing it.

    Args:
        client (GoogleAdsClient): An authenticated Google Ads API client created from credentials.
        query (str): The GAQL query string to validate (must target the configured customer/account).

    Process:
        1) Acquire the GoogleAdsService from the provided client.
        2) Create a SearchGoogleAdsStreamRequest and set customer_id and query.
        3) Invoke search_stream() with metadata flag (validate_only=true) so the API only validates.
        4) If no exception is raised, consider the query valid and return True.
        5) If a GoogleAdsException occurs, enumerate errors and return False.

    Returns:
        bool: True if the query is syntactically and semantically valid for the API, False otherwise.

    Error Handling:
        - GoogleAdsException: Captures and logs individual failure messages from the API.
        - Any other unexpected exception is not raised here (no generic except); only API errors are handled.

    Notes:
        - This call does not consume API quota for fetching data, but may still count toward request limits.
        - Useful to guard execution before running expensive queries.
    """
    try:
        ga_service = client.get_service("GoogleAdsService")
        request = client.get_type("SearchGoogleAdsStreamRequest")
        request.customer_id = customer_id
        request.query = query

        # Enable validation only by adding a metadata tuple
        ga_service.search_stream(request=request, metadata=[("validate_only", "true")])

        print("GAQL query validated successfully.")
        return True
    except GoogleAdsException as ex:
        print(" GAQL validation failed:")
        for error in ex.failure.errors:
            print(f"  - {error.message}")
        return False


In [0]:
"""
Fetches a report from the Google Ads API using a GAQL query and parses the response into a list of dictionaries. It handles nested objects, repeated fields, and enum values.
"""
from collections.abc import MutableSequence
from google.protobuf.descriptor import FieldDescriptor

def fetch_report_to_list(client: GoogleAdsClient, customer_id: str, query: str) -> list | None:
    """
    Execute a GAQL query via the Google Ads API and parse the streamed protobuf results
    into a list of flattened dictionaries.

    Args:
        client (GoogleAdsClient): Authenticated Google Ads client used to access services.
        customer_id (str): The customer account ID to query (no dashes, e.g., "1234567890").
        query (str): GAQL query string selecting fields from resource views.

    Process:
        1) Acquire GoogleAdsService and call search_stream(customer_id, query) to get a stream.
        2) Iterate over streamed batches and rows.
        3) For each protobuf row, recursively flatten all nested messages and repeated fields
           using the helper _parse_message().
        4) Collect each flattened row as a dict and append to a list.
        5) If the stream yields no rows, return None; otherwise return the list of dicts.

    Returns:
        list[dict] | None: A list of flattened row dictionaries, or None if no rows are returned.

    Error Handling:
        - GoogleAdsException: Logs the request ID and all error messages returned by the API,
          then returns None.
        - Generic Exception: Logs the unexpected error and returns None.

    Data Flattening Rules:
        - Nested messages are traversed depth-first and keys are joined with '.' (e.g., 'campaign.id').
        - Repeated fields (lists) are joined into a comma-separated string.
        - Enum fields are converted to their symbolic names when possible.
        - Field names with trailing underscores have the underscore removed for cleaner keys.
    """

    def _parse_message(message, parsed_row, prefix=""):
        """
        Recursively flatten a protobuf message into key/value pairs stored in parsed_row.

        Args:
            message: The protobuf message (row or nested message) to flatten.
            parsed_row (dict): Target dictionary mutated in place to accumulate flattened fields.
            prefix (str): Prefix to prepend to field names to reflect nesting (e.g., 'campaign.').

        Process:
            - Iterate over (field, value) pairs from message.ListFields().
            - Build a flattened key as f"{prefix}{field.name}" and normalize trailing underscores.
            - Handle value types:
              * MutableSequence: join entries by ", " and store as string.
              * Nested message: recurse with updated prefix "{key}.".
              * Enum: map numeric value to enum name when available; fallback to value.
              * Scalar: assign directly.

        Returns:
            None. Results are accumulated in parsed_row.

        Notes:
            - This helper is performance-sensitive; avoid heavy allocations inside loops.
        """
        for field, value in message.ListFields():
            key = f"{prefix}{field.name}"
            if key.endswith("_"):
                key = key[:-1]
            if isinstance(value, MutableSequence):
                parsed_row[key] = ", ".join(map(str, value))
            elif hasattr(value, "ListFields"):
                _parse_message(value, parsed_row, prefix=f"{key}.")
            elif field.type == FieldDescriptor.TYPE_ENUM:
                enum_type = field.enum_type
                enum_name = enum_type.values_by_number.get(value)
                parsed_row[key] = enum_name.name if enum_name else value
            else:
                parsed_row[key] = value

    try:
        logging.info(f"Executing query for customer_id: {customer_id}")
        google_ads_service = client.get_service("GoogleAdsService")
        stream = google_ads_service.search_stream(customer_id=customer_id, query=query)

        logging.info("Parsing API response stream...")
        parsed_rows = []

        for batch in stream:
            for row in batch.results:
                parsed_row = {}
                _parse_message(row._pb, parsed_row)
                parsed_rows.append(parsed_row)

        if not parsed_rows:
            logging.warning("Query executed successfully, but returned no rows.")
            return None

        logging.info(f"Successfully parsed {len(parsed_rows)} rows from the API.")
        return parsed_rows

    except GoogleAdsException as ex:
        logging.error(f"GAQL validation failed for Request ID '{ex.request_id}': {ex.error.code().name}")
        for error in ex.failure.errors:
            logging.error(f"\t- {error.message}")
        return None
    except Exception as e:
        logging.error(f"An unexpected Python error occurred during parsing: {e}")
        return None


In [0]:
"""
Defines the schema for each report type using PySpark's StructType. This is used to create DataFrames with the correct structure.
"""
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType,LongType,ArrayType

schema_map = {
    "core_campaign_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("campaign.status", StringType(), True),
        StructField("campaign.advertising_channel_type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "campaign_asset": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("campaign.status", StringType(), True),
        StructField("asset.id", StringType(), True),
        StructField("asset.type", StringType(), True),
        StructField("campaign_asset.status", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("segments.device", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "ad_group_ad_asset_view": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("ad_group_ad_asset_view.asset", StringType(), True),
        StructField("ad_group_ad_asset_view.field_type", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("segments.device", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "asset": StructType([
        StructField("asset.id", StringType(), True),
        StructField("asset.type", StringType(), True),
        StructField("asset.source", StringType(), True),
        StructField("asset.text_asset.text", StringType(), True),
        StructField("asset.image_asset.full_size.url", StringType(), True),
        StructField("asset.sitelink_asset.link_text", StringType(), True),
        StructField("asset.callout_asset.callout_text", StringType(), True),
        StructField("asset.structured_snippet_asset.header", StringType(), True),
        StructField("asset.structured_snippet_asset.values", StringType(), True),
    ]),

    "search_keyword_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group_criterion.criterion_id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("ad_group_criterion.keyword.text", StringType(), True),
        StructField("ad_group_criterion.keyword.match_type", StringType(), True),
        StructField("ad_group_criterion.quality_info.quality_score", IntegerType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
    ]),

    "search_term_analysis": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.name", StringType(), True),        
        StructField("search_term_view.search_term", StringType(), True),
        StructField("search_term_view.status", StringType(), True),
        StructField("segments.search_term_match_type", StringType(), True),
        StructField("segments.search_term_match_source", StringType(), True),
        StructField("segments.keyword.ad_group_criterion", StringType(), True),
        StructField("segments.keyword.info.text", StringType(), True),
        StructField("segments.keyword.info.match_type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
    ]),

    "conversion_action": StructType([
        StructField("conversion_action.id", StringType(), True),
        StructField("conversion_action.name", StringType(), True),
        StructField("conversion_action.status", StringType(), True),
        StructField("conversion_action.category", StringType(), True),
        StructField("conversion_action.primary_for_goal", StringType(), True),
        StructField("conversion_action.counting_type", StringType(), True),
        StructField("conversion_action.attribution_model_settings.attribution_model", StringType(), True),
        StructField("conversion_action.click_through_lookback_window_days", IntegerType(), True),
        StructField("conversion_action.view_through_lookback_window_days", IntegerType(), True),
        StructField("conversion_action.include_in_conversions_metric", StringType(), True),
    ]),

    "conversion_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("campaign.status", StringType(), True),
        StructField("segments.device", StringType(), True),
        StructField("segments.conversion_action", StringType(), True),
        StructField("segments.conversion_action_name", StringType(), True),
        StructField("segments.conversion_action_category", StringType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.conversions_value", DoubleType(), True),
        StructField("metrics.all_conversions", DoubleType(), True),
        StructField("metrics.all_conversions_value", DoubleType(), True),
        StructField("metrics.value_per_conversion", DoubleType(), True),
        StructField("metrics.value_per_all_conversions", DoubleType(), True),
    ]),

    "ad_copy_and_landing_page_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group_ad.ad.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("ad_group_ad.ad.final_urls", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
    ]),

    "optimization_device_and_time": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("segments.device", StringType(), True),
        StructField("segments.day_of_week", StringType(), True),
        StructField("segments.hour", IntegerType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "placement_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("campaign.advertising_channel_type", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("detail_placement_view.placement", StringType(), True),
        StructField("detail_placement_view.placement_type", StringType(), True),
        StructField("detail_placement_view.display_name", StringType(), True),
        StructField("detail_placement_view.group_placement_target_url", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "group_placement_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("campaign.advertising_channel_type", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("group_placement_view.placement", StringType(), True),
        StructField("group_placement_view.placement_type", StringType(), True),
        StructField("group_placement_view.display_name", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
        StructField("metrics.average_cpc", DoubleType(), True),
    ]),

    "audience_gender_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group_criterion.criterion_id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group_criterion.gender.type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.conversions", DoubleType(), True),
    ]),

    "audience_age_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group_criterion.criterion_id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group_criterion.age_range.type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
    ]),

    "competitive_impression_share": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("metrics.search_impression_share", DoubleType(), True),
        StructField("metrics.search_top_impression_share", DoubleType(), True),
        StructField("metrics.search_absolute_top_impression_share", DoubleType(), True),
    ]),

    "network_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("segments.ad_network_type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
    ]),

    "display_ad_viewability": StructType([
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("metrics.active_view_impressions", IntegerType(), True),
        StructField("metrics.active_view_measurability", DoubleType(), True),
        StructField("metrics.active_view_viewability", DoubleType(), True),
        StructField("metrics.active_view_cpm", DoubleType(), True),
    ]),

    "Call_lead_generation": StructType([
        StructField("segments.date", DateType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("metrics.phone_calls", IntegerType(), True),
        StructField("metrics.phone_impressions", IntegerType(), True),
        StructField("metrics.phone_through_rate", DoubleType(), True),
    ]),

    "geographic_performance": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("segments.date", DateType(), True),
        StructField("geographic_view.country_criterion_id", StringType(), True),
        StructField("segments.geo_target_city", StringType(), True),
        StructField("geographic_view.location_type", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.cost_micros", DoubleType(), True),
        StructField("metrics.ctr", DoubleType(), True),
    ]),

    "ads_performance": StructType([
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group_ad.ad.id", StringType(), True),
        StructField("metrics.impressions", IntegerType(), True),
        StructField("metrics.clicks", IntegerType(), True),
        StructField("metrics.conversions", DoubleType(), True),
        StructField("metrics.cost_micros", LongType(), True)
    ]),

    "ads_data": StructType([
        StructField("campaign.id", StringType(), True),
        StructField("campaign.name", StringType(), True),
        StructField("ad_group.id", StringType(), True),
        StructField("ad_group.name", StringType(), True),
        StructField("ad_group_ad.ad.id", StringType(), True),
        StructField("ad_group_ad.ad.type", StringType(), True),
        StructField("ad_group_ad.status", StringType(), True),
        StructField("ad_group_ad.ad.final_urls", StringType(), True),
        StructField("ad_group_ad.ad.responsive_search_ad.headlines", StringType(), True),
        StructField("ad_group_ad.ad.responsive_search_ad.descriptions", StringType(), True),
        ##suggested type change by chatgpt below but not updated yet - for review
        #StructField("ad_group_ad.ad.responsive_search_ad.headlines", ArrayType(StringType()), True),
        #StructField("ad_group_ad.ad.responsive_search_ad.descriptions", ArrayType(StringType()), True),
        StructField("ad_group_ad.ad.expanded_text_ad.headline_part1", StringType(), True),
        StructField("ad_group_ad.ad.expanded_text_ad.headline_part2", StringType(), True),
        StructField("ad_group_ad.ad.expanded_text_ad.headline_part3", StringType(), True),
        StructField("ad_group_ad.ad.expanded_text_ad.description", StringType(), True),
        StructField("ad_group_ad.ad.expanded_text_ad.description2", StringType(), True),
        StructField("ad_group_ad.ad.responsive_display_ad.long_headline", StringType(), True),
        StructField("ad_group_ad.ad.responsive_display_ad.headlines", StringType(), True),
        StructField("ad_group_ad.ad.responsive_display_ad.descriptions", StringType(), True),
        ##suggested type change by chatgpt below but not updated yet - for review
        #StructField("ad_group_ad.ad.responsive_display_ad.headlines", ArrayType(StringType()), True),
        #StructField("ad_group_ad.ad.responsive_display_ad.descriptions", ArrayType(StringType()), True),
        StructField("ad_group_ad.ad.image_ad.image_url", StringType(), True),
        StructField("ad_group_ad.ad.image_ad.mime_type", StringType(), True),
        ##Deprecated columns in googleads v23
        #StructField("ad_group_ad.ad.call_ad.business_name", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.country_code", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.phone_number", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.headline1", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.headline2", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.description1", StringType(), True),
        #StructField("ad_group_ad.ad.call_ad.description2", StringType(), True),
    ]),

    "geo_target_constant": StructType([
        StructField("geo_target_constant.id", StringType(), True),
        StructField("geo_target_constant.name", StringType(), True),
        StructField("geo_target_constant.canonical_name", StringType(), True),
        StructField("geo_target_constant.country_code", StringType(), True),
        StructField("geo_target_constant.target_type", StringType(), True),
    ])
}

In [0]:
"""
Safely casts values to the desired type, returning None if the cast fails. Used to normalize data types when processing API responses.
"""
import logging
from datetime import datetime

def safe_cast(value, cast_type):
    """
    Safely cast a value to a target type, returning None if the conversion fails.

    Args:
        value: Any value to cast (may be None, str, int, float, etc.).
        cast_type (Callable): A callable that converts the input to the desired type (e.g., int, float, str).

    Process:
        - If value is None, immediately return None.
        - Attempt to call cast_type(value).
        - Catch ValueError and TypeError and return None on failure.

    Returns:
        Any | None: The converted value on success; None if value is None or casting fails.

    Examples:
        safe_cast("123", int) -> 123
        safe_cast("abc", int) -> None
        safe_cast(None, float) -> None
    """
    try:
        if value is None:
            return None
        return cast_type(value)
    except (ValueError, TypeError):
        return None

"""
Normalizes and casts a record's fields to the appropriate types based on the schema definition. This ensures consistency and correctness of data types.
"""
def normalize_and_cast(record: dict, schema):
    """
    Normalize and type-cast a flat record according to a provided Spark StructType schema.

    Args:
        record (dict): A dictionary of string keys to raw values (e.g., parsed API row).
        schema (StructType): The Spark schema whose fields (name and dataType) drive casting logic.

    Process:
        1) Initialize an empty dict `normalized`.
        2) For each field in schema.fields, read the raw value from record by field.name.
        3) Based on the field's dataType:
           - integer -> cast with safe_cast(value, int)
           - double  -> cast with safe_cast(value, float)
           - date    -> parse YYYY-MM-DD to date; on failure, warn and set None
           - other   -> keep original value as-is
        4) Store the converted value in normalized[field.name].

    Returns:
        dict: A dictionary guaranteed to have all schema field names with best-effort typed values.

    Side Effects:
        - Emits a warning log if date parsing fails for any field.

    Error Handling:
        - Uses safe_cast to avoid raising on numeric conversion errors.
        - Wraps date parsing in try/except and falls back to None.
    """
    normalized = {}
    for field in schema.fields:
        key = field.name
        raw_value = record.get(key)

        if field.dataType.typeName() == "integer":
            normalized[key] = safe_cast(raw_value, int)
        elif field.dataType.typeName() == "double":
            normalized[key] = safe_cast(raw_value, float)
        elif field.dataType.typeName() == "date":
            try:
                normalized[key] = datetime.strptime(raw_value, "%Y-%m-%d").date() if raw_value else None
            except Exception:
                logging.warning(f"⚠️ Failed to parse date for field '{key}' (value: {raw_value})")
                normalized[key] = None
        else:
            normalized[key] = raw_value
    return normalized



# Normal Bronze processing loop
"""
Main processing loop: Runs each GAQL query, validates, fetches, normalizes, creates DataFrames, displays samples, and saves to Delta tables in the bronze layer.
"""
for name, query in query_map.items():
    print(f"\n🚀 Running GAQL for: {name}")

    if not validate_gaql(client, query):
        logging.warning(f"⚠️ Skipping invalid GAQL query: {name}")
        continue

    data = fetch_report_to_list(client, customer_id, query)

    if data:
        schema = schema_map.get(name)
        if schema:
            normalized_data = [normalize_and_cast(r, schema) for r in data]
            try:
                globals()[f"{name}_df"] = spark.createDataFrame(normalized_data, schema=schema)
                logging.info(f"✨ Successfully created Spark DataFrame: {name}_df")
            except Exception as e:
                logging.error(f"❌ Schema mismatch for '{name}': {e}")
                continue
        else:
            globals()[f"{name}_df"] = spark.createDataFrame(data)
            logging.info(f"ℹ️ Created DataFrame with inferred schema: {name}_df")

        display(globals()[f"{name}_df"].limit(5))
        globals()[f"{name}_df"].write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable(f"googleads_bronze.{name}")

    else:
        logging.warning(f"⚠️ No data was returned for '{name}'.")


🚀 Running GAQL for: core_campaign_performance


2026-03-03 06:46:08,671 - INFO - Executing query for customer_id: 1401815809


GAQL query validated successfully.


2026-03-03 06:46:09,437 - INFO - Parsing API response stream...
2026-03-03 06:46:09,450 - INFO - Request made: ClientCustomerId: 1401815809, Host: googleads.googleapis.com, Method: /google.ads.googleads.v23.services.GoogleAdsService/SearchStream, RequestId: M19iJcMR_p5RD94rmsKn-g, IsFault: False, FaultMessage: None
2026-03-03 06:46:09,450 - INFO - Successfully parsed 340 rows from the API.
2026-03-03 06:46:09,797 - INFO - ✨ Successfully created Spark DataFrame: core_campaign_performance_df


campaign.id,segments.date,campaign.name,campaign.status,campaign.advertising_channel_type,metrics.impressions,metrics.clicks,metrics.ctr,metrics.cost_micros,metrics.average_cpc
21771245709,2024-11-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,112056,1179,0.010521524951809809,3.92312256E8,332750.00508905854
22973632076,2025-09-08,Data Engineering (AWS-Sept),PAUSED,PERFORMANCE_MAX,34546,1422,0.04116250796040063,1.381432353E9,971471.4156118144
21771245709,2025-12-22,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,22702,3060,0.13478988635362524,9.93128612E8,324551.8339869281
21771245709,2025-02-06,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,21020,2364,0.11246431969552807,3.93109118E8,166289.8130287648
21771245709,2024-11-26,Azure Data Engineering -Display - Image,PAUSED,DISPLAY,20317,1092,0.05374809273022592,4.28549073E8,392444.20604395604


...


🚀 Running GAQL for: geo_target_constant


2026-03-03 06:48:33,500 - INFO - Executing query for customer_id: 1401815809


GAQL query validated successfully.


2026-03-03 06:48:33,997 - INFO - Parsing API response stream...
2026-03-03 06:48:37,140 - INFO - Request made: ClientCustomerId: 1401815809, Host: googleads.googleapis.com, Method: /google.ads.googleads.v23.services.GoogleAdsService/SearchStream, RequestId: 40iZbiR6SGDrGQFPd9cBKQ, IsFault: False, FaultMessage: None
2026-03-03 06:48:37,141 - INFO - Successfully parsed 228994 rows from the API.
2026-03-03 06:48:38,736 - INFO - ✨ Successfully created Spark DataFrame: geo_target_constant_df


geo_target_constant.id,geo_target_constant.name,geo_target_constant.canonical_name,geo_target_constant.country_code,geo_target_constant.target_type
2004,Afghanistan,Afghanistan,AF,Country
2008,Albania,Albania,AL,Country
2010,Antarctica,Antarctica,AQ,Country
2012,Algeria,Algeria,DZ,Country
2016,American Samoa,American Samoa,AS,Country
